## Preprocessing: Stage 2 of the pipeline

This script turns the raw scraped records into a clean, and stratified test/train split. Runs before feature engineering and only touches lightweight record metadata. 
### Roles:
1. Load and validate raw JSON records
2. Deduplicate by label and hostname: guards against the same domain being pulled in by more than one overlapping phishing feed and inflating that domain's weight in the dataset.
3. Stratified 80/20 train-test split, writing a manifest and per-split lists. Stratification alone preserves the exact class ratio in both splits.

In [45]:
# Library imports
import glob
import json
import logging
import os
import sys
import pandas as pd
from sklearn.model_selection import train_test_split

In [46]:
# Directory mapping
RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
LOG_PATH = "logs/preprocessed.log"
manifest_path = os.path.join(PROCESSED_DIR, "dataset_manifest.csv")

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs("logs", exist_ok=True)

In [47]:
# Logging configuration
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.flush = sys.stdout.flush
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(LOG_PATH), stream_handler],
    force=True,
)
log = logging.getLogger(__name__)

In [48]:
# Required fields (global schema)
REQUIRED_FIELDS = ["sample_id", "label", "source", "seed_url", 
                   "final_url", "hostname", "http_status", "raw_html"]

### Loading and Validation

In [49]:
progress_every = 2000 # report processing processing after every 2000
                      # processed files

paths = glob.glob(os.path.join(RAW_DIR, "*.json"))  # load records
log.info(f"Total loaded records: {len(paths)}")


2026-09-09 21:10:52,212 [INFO] Total loaded records: 17250


In [50]:
# checking for corrupt rows and records with missing fields
rows, n_corrupt, n_misisng_fieds = [], 0, 0

for i, p in enumerate(paths, start=1):
    try:
        with open(p, "r", encoding="utf-8") as f:
            rec = json.load(f)
    except (json.JSONDecodeError, UnicodeDecodeError, OSError) as e:
        log.warning(f"[corrupt] could not parse -> {e}")
        n_corrupt += 1
        continue

    missing = [field for field in REQUIRED_FIELDS if field not in rec or rec[field] in (None, "")]

    if missing:
        log.warning(f"[incomplete] {p} missing fields: {missing} -- DROPPED")
        n_misisng_fieds += 1
        continue

    rows.append({
        "sample_id": rec["sample_id"],
        "url": rec["final_url"],
        "source": rec["source"],
        "hostname": rec["hostname"],
        "http_status": rec["http_status"],
        "n_inline_scripts": len(rec.get("inline_scripts", [])),
        "n_external_scripts": len(rec.get("external_script_srcs", [])),
        "html_length": len(rec.get("raw_html", "")),
        "label": 1 if rec["label"] == "phishing" else 0
    })

    if progress_every and i % progress_every == 0:
        log.info(f"Processed {i}/{len(paths)} raw files")

log.info(
    f"Loaded {len(rows)} valid records "
    f"({n_corrupt} corrupt, {n_misisng_fieds} missing required fields discarded)"
)

2026-09-09 21:10:52,392 [WARNING] [incomplete] data/raw\02eeb4d5f41b7871.json missing fields: ['raw_html'] -- DROPPED
2026-09-09 21:10:52,417 [WARNING] [incomplete] data/raw\036ce0d57bd5f223.json missing fields: ['raw_html'] -- DROPPED
2026-09-09 21:10:52,420 [WARNING] [incomplete] data/raw\0373ce7fb4fe4966.json missing fields: ['raw_html'] -- DROPPED
2026-09-09 21:10:52,682 [WARNING] [incomplete] data/raw\056cf1f8efa9924e.json missing fields: ['raw_html'] -- DROPPED
2026-09-09 21:10:52,688 [WARNING] [incomplete] data/raw\05800aee5d7cd429.json missing fields: ['raw_html'] -- DROPPED
2026-09-09 21:10:52,819 [WARNING] [incomplete] data/raw\06e647d2f1594e58.json missing fields: ['raw_html'] -- DROPPED
2026-09-09 21:10:53,524 [WARNING] [incomplete] data/raw\0e434a5d76861f64.json missing fields: ['raw_html'] -- DROPPED
2026-09-09 21:10:53,537 [WARNING] [incomplete] data/raw\0e5208649bc61634.json missing fields: ['raw_html'] -- DROPPED
2026-09-09 21:10:53,800 [WARNING] [incomplete] data/raw\

### Deduplication

In [51]:
df = pd.DataFrame(rows)
df.head()

,sample_id,url,source,hostname,http_status,n_inline_scripts,n_external_scripts,html_length,label
0,00042be843942e60,https://www.google.com,phishtank,www.google.com,200,13,2,215808,1
1,000f68d47c340467,https://name-servers.gr/,tranco,name-servers.gr,200,0,0,841,0
2,00104681a63fc3b9,http://0efbd9f.wcomhost.com/de/banking-postban...,phishing_db,0efbd9f.wcomhost.com,200,0,0,122,1
3,0011ddaade4dcf56,https://www.eskimi.com/,tranco,www.eskimi.com,200,24,12,173368,0
4,0019403344542c89,https://docs.google.com/drawings/d/1gWjHZT7bNp...,phishtank,docs.google.com,200,31,2,170779,1


In [52]:
dedup_df = df.drop_duplicates(subset=["label", "hostname"], keep="first").reset_index(drop=True)
log.info(
    f"Before deduplication: {len(df)} "
    f"After deduplication: {len(dedup_df)}"
    )

2026-09-09 21:11:13,972 [INFO] Before deduplication: 17139 After deduplication: 11211


In [53]:
label_counts = dedup_df["label"].value_counts()
log.info(
    f"[reporting class balance] \n{label_counts}"
)

def report_balance(df, stage_label):
    counts = df["label"].value_counts()
    log.info(f"Class label [{stage_label}]: " + ", ".join(f"{k}={v}" for k, v in counts.items()))

2026-09-09 21:11:13,978 [INFO] [reporting class balance] 
label
0    6853
1    4358
Name: count, dtype: int64


### Dataset splits

In [54]:
train_df, test_df = train_test_split(dedup_df, test_size=0.2, random_state=42,
                                     stratify=dedup_df["label"])
report_balance(train_df, "train split")
report_balance(test_df, "test split")

2026-09-09 21:11:13,990 [INFO] Class label [train split]: 0=5482, 1=3486
2026-09-09 21:11:13,991 [INFO] Class label [test split]: 0=1371, 1=872


In [55]:
# save clean dataset as csv
train_df = train_df.assign(split="train")
test_df = test_df.assign(split="test")
full_df = pd.concat([train_df, test_df]).reset_index(drop=True)

full_df.to_csv(manifest_path, index=False)
log.info(f"Wrote dataset manifest -> {manifest_path} {len(full_df)} rows")

# write train-test ids
for name, part in [("train", train_df), ("test", test_df)]:
    ids_path = os.path.join(PROCESSED_DIR, f"{name}_ids.txt")
    with open(ids_path, "w") as f:
        f.write("\n".join(part["sample_id"].tolist()))
    log.info(f"Wrote {name} split IDs -> ids_path ({len(part)} ids)")

2026-09-09 21:11:14,024 [INFO] Wrote dataset manifest -> data/processed\dataset_manifest.csv 11211 rows
2026-09-09 21:11:14,027 [INFO] Wrote train split IDs -> ids_path (8968 ids)
2026-09-09 21:11:14,028 [INFO] Wrote test split IDs -> ids_path (2243 ids)
